# Spark Sandbox — Getting Started

## Service endpoints

| Service | URL | Notes |
|---|---|---|
| Spark Master UI | http://localhost:8080 | Cluster overview |
| Spark App UI | http://localhost:4040 | Active while a job runs |
| Kafka UI | http://localhost:8090 | Topics, messages, consumer groups |
| JupyterLab | http://localhost:8888 | token: `spark123` |
| PostgreSQL | `localhost:5432` | db: sandbox, user: spark, pwd: spark123 |

## Sections
1. SparkSession setup
2. Batch pipeline → PostgreSQL
3. DuckDB analytics on Parquet
4. Kafka producer
5. Spark Structured Streaming from Kafka

---
## 1 · SparkSession

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

SPARK_MASTER  = os.getenv('SPARK_MASTER',            'spark://spark-master:7077')
KAFKA_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:29092')
PG_URL        = f"jdbc:postgresql://{os.getenv('POSTGRES_HOST','postgres')}:5432/{os.getenv('POSTGRES_DB','sandbox')}"
PG_PROPS      = {'user': os.getenv('POSTGRES_USER','spark'), 'password': os.getenv('POSTGRES_PASSWORD','spark123'), 'driver': 'org.postgresql.Driver'}

spark = (
    SparkSession.builder
    .appName('sandbox')
    .master(SPARK_MASTER)
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} — master: {spark.sparkContext.master}')

---
## 2 · Batch pipeline

Pattern: **generate → transform → write PostgreSQL + Parquet**

In [ ]:
# Read existing tables from PostgreSQL
users = spark.read.jdbc(PG_URL, 'users', properties=PG_PROPS)
txns  = spark.read.jdbc(PG_URL, 'transactions', properties=PG_PROPS)

users.printSchema()
txns.show(5)

In [ ]:
from faker import Faker
import random

# Generate synthetic transactions
fake   = Faker()
CATS   = ['electronics', 'food', 'clothing', 'travel', 'health']
STATUS = ['completed', 'pending', 'refunded']

rows = [
    (random.randint(1, 5), round(random.uniform(5, 500), 2),
     random.choice(CATS), random.choice(STATUS),
     fake.date_time_between('-30d', 'now'))
    for _ in range(200)
]

schema = StructType([
    StructField('user_id',    IntegerType()),
    StructField('amount',     DoubleType()),
    StructField('category',   StringType()),
    StructField('status',     StringType()),
    StructField('created_at', TimestampType()),
])

df = spark.createDataFrame(rows, schema)

# Enrich with user country
enriched = (
    df.join(users.select('id', 'country'), df.user_id == users.id)
      .drop('id')
      .withColumn('date', F.to_date('created_at'))
)

enriched.show(5)

In [ ]:
# Daily category summary
summary = (
    enriched
    .filter(F.col('status') == 'completed')
    .groupBy('date', 'category')
    .agg(
        F.round(F.sum('amount'), 2).alias('total_amount'),
        F.count('*').alias('tx_count'),
    )
    .orderBy('date', 'category')
)

summary.show(10)

In [ ]:
# Write to PostgreSQL
summary.withColumnRenamed('date', 'summary_date') \
       .write.jdbc(PG_URL, 'daily_summary', mode='append', properties=PG_PROPS)

# Write to Parquet (for DuckDB section)
enriched.write.mode('overwrite').parquet('/home/jupyter/data/output/transactions')

print('Done writing.')

---
## 3 · DuckDB analytics

DuckDB can query Parquet files directly — great for fast ad-hoc analytics after a Spark job.

In [ ]:
import duckdb

con = duckdb.connect()  # in-memory

# Register the Parquet dataset written by Spark
con.execute("CREATE VIEW transactions AS SELECT * FROM read_parquet('/home/jupyter/data/output/transactions/**/*.parquet')")

con.execute('SELECT * FROM transactions LIMIT 5').df()

In [ ]:
# Revenue by country & category
result = con.execute("""
    SELECT
        country,
        category,
        ROUND(SUM(amount), 2)  AS total_revenue,
        COUNT(*)               AS tx_count,
        ROUND(AVG(amount), 2)  AS avg_amount
    FROM transactions
    WHERE status = 'completed'
    GROUP BY ALL
    ORDER BY total_revenue DESC
""").df()

result

In [ ]:
import matplotlib.pyplot as plt

pivot = result.pivot(index='category', columns='country', values='total_revenue').fillna(0)
pivot.plot(kind='bar', figsize=(10, 5), title='Revenue by Category & Country')
plt.tight_layout()
plt.show()

---
## 4 · Kafka producer

Send simulated events that the streaming job will consume.

In [ ]:
import json, time, random, threading
from kafka import KafkaProducer
from faker import Faker
from datetime import datetime, timezone

fake = Faker()
CATS = ['electronics', 'food', 'clothing', 'travel', 'health']

producer = KafkaProducer(
    bootstrap_servers=KAFKA_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode(),
)

def send_events(n=50, delay=0.2):
    for i in range(n):
        event = {
            'event_id':  fake.uuid4(),
            'user_id':   random.randint(1, 5),
            'amount':    round(random.uniform(5, 500), 2),
            'category':  random.choice(CATS),
            'ts':        datetime.now(timezone.utc).isoformat(),
        }
        producer.send('transactions', value=event)
        time.sleep(delay)
    producer.flush()
    print(f'Sent {n} events.')

# Send events in a background thread so the notebook stays responsive
t = threading.Thread(target=send_events, args=(100, 0.1), daemon=True)
t.start()
print('Producer started — check Kafka UI at http://localhost:8090')

---
## 5 · Spark Structured Streaming

Consume `transactions` topic, aggregate over a 1-minute tumbling window, write to console and PostgreSQL.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

event_schema = StructType([
    StructField('event_id', StringType()),
    StructField('user_id',  IntegerType()),
    StructField('amount',   DoubleType()),
    StructField('category', StringType()),
    StructField('ts',       StringType()),
])

raw = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_SERVERS)
    .option('subscribe', 'transactions')
    .option('startingOffsets', 'earliest')
    .load()
)

events = (
    raw
    .select(F.from_json(F.col('value').cast('string'), event_schema).alias('e'))
    .select('e.*')
    .withColumn('event_time', F.to_timestamp('ts'))
)

events.printSchema()

In [ ]:
# 1-minute tumbling window aggregation with 10-second watermark
windowed = (
    events
    .withWatermark('event_time', '10 seconds')
    .groupBy(
        F.window('event_time', '1 minute'),
        'category',
    )
    .agg(
        F.round(F.sum('amount'), 2).alias('total_amount'),
        F.count('*').alias('tx_count'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'category',
        'total_amount',
        'tx_count',
    )
)

In [ ]:
# Print to console — swap outputMode/sink for production use
query = (
    windowed.writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('checkpointLocation', '/home/jupyter/data/checkpoints/streaming_demo')
    .trigger(processingTime='15 seconds')
    .start()
)

print(f'Streaming query started: {query.id}')
print('Results appear in the Spark UI → Structured Streaming tab.')
print('Run query.stop() to halt.')

In [ ]:
# Stop when done
query.stop()
print('Stream stopped.')